# PanVision simple CNN — trained from scratch

A four-block convolutional baseline: Conv-BatchNorm-ReLU-MaxPool with 32, 64, 128,
256 channels → global average pooling → 512 features → dropout 0.3 → 2 class logits.
No pretrained weights are used. Uses exactly the same supplied crops, shared patient
splits, augmentation and patient-level scoring as the DenseNet notebook. Input images
remain three-channel grayscale with the same normalization for a controlled baseline.

**Scope:** annotation-assisted exploratory classification. Crop generation consults
reference lesion masks, including held-out slices. These results cannot establish
performance on ordinary uploaded CT scans. Test patients are not used for selection.

Start with the optional smoke test, then run eight epochs. A scratch CNN may need a
different learning rate or more epochs; tune on validation only, document all changes,
and do not assume eight epochs gives equally converged models. Checkpoints use a
separate `simplecnn_...` Google Drive folder and never overwrite DenseNet or ResNet.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scikit-learn', 'pillow'])
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime before training.')
print(torch.cuda.get_device_name(0))


## 1. Set your Google Drive crops path
Upload the extracted **crops** folder (or the completed crops ZIP) to your Drive.
The expected default is **My Drive / PanVision / crops**. Edit `DATA_SOURCE` below
if yours is elsewhere. Crops are copied to `/content` for faster training; checkpoints
are saved to Drive. Drive authorization is handled by Colab.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
# Set this to your crops folder OR finished ZIP in Google Drive.
DATA_SOURCE = Path('/content/drive/MyDrive/PanVision/crops')
if not DATA_SOURCE.exists():
    raise FileNotFoundError(f'Update DATA_SOURCE to your uploaded crops folder/ZIP: {DATA_SOURCE}')


In [ ]:
import zipfile, shutil, hashlib
CROPS = Path('/content/panvision_crops')
if CROPS.exists():
    raise RuntimeError('Extraction directory already exists. Use a fresh runtime or a new directory.')
if DATA_SOURCE.is_dir():
    originals = sorted(DATA_SOURCE.rglob('*.npz'))
    names = [p.name for p in originals]
    if not originals or len(names) != len(set(names)):
        raise ValueError('No crops or duplicate crop filenames')
    if sum(p.stat().st_size for p in originals) > shutil.disk_usage('/content').free - 5*1024**3:
        raise RuntimeError('Insufficient local disk space')
    CROPS.mkdir()
    for original in originals:
        target = CROPS / original.name
        shutil.copyfile(original, target)
        with original.open('rb') as f:
            expected = hashlib.file_digest(f, 'sha256').hexdigest()
        with target.open('rb') as f:
            actual = hashlib.file_digest(f, 'sha256').hexdigest()
        if expected != actual:
            raise ValueError(f'Copy mismatch: {original.name}')
    print('Copied verified crop files:', len(originals))
else:
    with zipfile.ZipFile(DATA_SOURCE) as archive:
        members = [m for m in archive.infolist()
                   if m.filename.endswith('.npz') and not m.filename.startswith('__MACOSX/')]
        if not members:
            raise ValueError('No NPZ crop files found; inspect the archive format before continuing.')
        names = [Path(m.filename).name for m in members]
        if len(names) != len(set(names)):
            raise ValueError('Duplicate crop filenames in ZIP.')
        total = sum(m.file_size for m in members)
        if total > shutil.disk_usage('/content').free - 5 * 1024**3:
            raise RuntimeError('Insufficient local space for extraction plus reserve.')
        CROPS.mkdir()
        for m in members:
            # Ignore archive paths: extract NPZ data only to flat, unique basenames.
            with archive.open(m) as source, (CROPS / Path(m.filename).name).open('xb') as target:
                shutil.copyfileobj(source, target)
    print('Extracted NPZ files:', len(members))


## 2. Install the supplied model and loader code into this runtime

In [ ]:
import json, sys
WORK = Path('/content/panvision_densenet_classifier')
WORK.mkdir(exist_ok=True)
SOURCES = {'dataset.py': '"""Feed the saved 2D crops to a PyTorch model. Shared by everyone training a classifier.\n\nRead the crops made by make_crops.py and hand them over one picture at a time.\nEvery picture comes with the patient\'s label (1 = PDAC) and a patient number, so the\nscores of one patient\'s slices can be averaged afterwards.\n\nmode="train": PDAC patients contribute only slices that contain tumour (their other\n              slices look healthy, so the label "PDAC" would be misleading there).\n              Non-PDAC patients contribute all their saved pancreas slices.\nmode="eval":  every patient contributes all of their saved pancreas slices, because at\n              test time we do not know where the tumour is.\n\nUse it like this:\n    train = CropDataset("outputs/crops", split="train", mode="train", augment=True)\n    val   = CropDataset("outputs/crops", split="val",   mode="eval")\n    loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)\n"""\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\n# Standard values the ImageNet-trained Simple CNN expects (same as the explainability code).\nMEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\nSTD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\n\n\ndef select_rows(has_lesion, label, mode, drop_unlocalised_pdac=False):\n    """Which of a study\'s saved slices to use. Returns row numbers (maybe empty)."""\n    has_lesion = np.asarray(has_lesion, dtype=bool)\n    if mode == "train" and label == 1:\n        if has_lesion.any():\n            return np.where(has_lesion)[0]\n        if drop_unlocalised_pdac:\n            return np.array([], dtype=int)  # cannot tell which slices show tumour\n    return np.arange(len(has_lesion))\n\n\ndef load_cache(cache_dir, split, mode, drop_unlocalised_pdac=False):\n    """Read the saved crops for one split. Returns plain numpy arrays (no torch)."""\n    images, labels, patient_idx, study_ids = [], [], [], []\n    patient_ids, patient_labels = [], []\n    for path in sorted(Path(cache_dir).glob("*.npz")):\n        if path.name.endswith(".tmp.npz"):\n            continue\n        with np.load(path) as f:\n            if split != "all" and str(f["split"]) != split:\n                continue\n            label = int(f["label"])\n            rows = select_rows(f["has_lesion"], label, mode, drop_unlocalised_pdac)\n            if len(rows) == 0:\n                continue\n            pid = str(f["patient_id"])\n            if pid not in patient_ids:\n                patient_ids.append(pid)\n                patient_labels.append(label)\n            images.append(f["crops"][rows])\n            labels += [label] * len(rows)\n            patient_idx += [patient_ids.index(pid)] * len(rows)\n            study_ids += [str(f["study_id"])] * len(rows)\n    if not images:\n        raise FileNotFoundError(f"No crops found for split \'{split}\' in {cache_dir}")\n    return {\n        "images": np.concatenate(images),\n        "labels": np.array(labels),\n        "patient_idx": np.array(patient_idx),\n        "study_ids": np.array(study_ids),\n        "patient_ids": np.array(patient_ids),\n        "patient_labels": np.array(patient_labels),\n    }\n\n\nclass CropDataset(Dataset):\n    def __init__(self, cache_dir, split, mode="train", augment=False, drop_unlocalised_pdac=False):\n        data = load_cache(cache_dir, split, mode, drop_unlocalised_pdac)\n        self.images = data["images"]  # (N, 224, 224) whole numbers 0-255, kept in memory\n        self.labels = data["labels"]\n        self.patient_idx = data["patient_idx"]  # which patient each picture belongs to\n        self.study_ids = data["study_ids"]\n        self.patient_ids = data["patient_ids"]\n        self.patient_labels = data["patient_labels"]\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.images)\n\n    def __getitem__(self, i):\n        x = torch.from_numpy(self.images[i]).float() / 255.0  # (224, 224), values 0-1\n        if self.augment:\n            x = self._jitter(x)\n        x = x.unsqueeze(0).repeat(3, 1, 1)  # models expect 3 colour channels\n        x = (x - MEAN) / STD\n        return x, int(self.labels[i]), int(self.patient_idx[i])\n\n    @staticmethod\n    def _jitter(x, max_shift=8):\n        """Small random brightness/contrast change and a few pixels of shift.\n\n        No flips or rotations: left/right and up/down mean something anatomically.\n        """\n        contrast = 0.9 + 0.2 * torch.rand(1)\n        brightness = -0.05 + 0.1 * torch.rand(1)\n        x = ((x - 0.5) * contrast + 0.5 + brightness).clamp(0, 1)\n        padded = torch.nn.functional.pad(\n            x[None, None], (max_shift,) * 4, mode="replicate")[0, 0]\n        dy, dx = torch.randint(0, 2 * max_shift + 1, (2,)).tolist()\n        return padded[dy:dy + x.shape[0], dx:dx + x.shape[1]]\n\n    def pos_weight(self):\n        """How much more to weight PDAC pictures so the 30% minority is not ignored."""\n        positives = float((self.labels == 1).sum())\n        return float((len(self.labels) - positives) / max(positives, 1.0))\n', 'evaluation.py': '"""Shared scoring, so everyone who trains a classifier reports the SAME numbers.\n\nThe label belongs to a patient, not to a slice. So the model\'s scores for all of one\npatient\'s slices are averaged into ONE score per patient, and everything below is\nmeasured per patient.\n\nRules for a fair comparison:\n  * choose settings (epochs, threshold ...) on the VALIDATION patients only;\n  * look at the TEST patients once, at the very end.\n"""\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score, roc_curve\n\n\ndef patient_scores(probs, patient_idx, patient_labels):\n    """Average slice scores into one score per patient.\n\n    probs: PDAC probability of every picture;  patient_idx: patient number of every picture;\n    patient_labels: label (0/1) of every patient. Returns (score per patient, label per patient).\n    """\n    probs = np.asarray(probs, dtype=float)\n    patient_idx = np.asarray(patient_idx, dtype=int)\n    patient_labels = np.asarray(patient_labels, dtype=int)\n    n = len(patient_labels)\n    sums = np.bincount(patient_idx, weights=probs, minlength=n)\n    counts = np.bincount(patient_idx, minlength=n)\n    seen = counts > 0  # ignore any patient who has no pictures\n    return sums[seen] / counts[seen], patient_labels[seen]\n\n\ndef pick_threshold(scores, labels):\n    """The cut-off that best separates the classes (largest sensitivity + specificity - 1).\n\n    Choose it on the validation patients, then keep it fixed for the test patients.\n    """\n    fpr, tpr, thresholds = roc_curve(labels, scores)\n    best = int(np.argmax(tpr - fpr))\n    return float(thresholds[best])\n\n\ndef evaluate_patients(probs, patient_idx, patient_labels, threshold=0.5):\n    """Per-patient results: AUC, sensitivity, specificity, accuracy.\n\n    AUC: 0.5 = coin flip, 1.0 = perfect. Sensitivity = share of PDAC patients caught.\n    Specificity = share of non-PDAC patients correctly cleared.\n    """\n    scores, y = patient_scores(probs, patient_idx, patient_labels)\n    predicted = scores >= threshold\n    positives, negatives = y == 1, y == 0\n    return {\n        "n_patients": int(len(y)),\n        "n_pdac": int(positives.sum()),\n        "auc": float(roc_auc_score(y, scores)) if positives.any() and negatives.any() else float("nan"),\n        "sensitivity": float(predicted[positives].mean()) if positives.any() else float("nan"),\n        "specificity": float((~predicted)[negatives].mean()) if negatives.any() else float("nan"),\n        "accuracy": float((predicted == (y == 1)).mean()),\n        "threshold": float(threshold),\n    }\n\n\ndef format_metrics(m):\n    return (f"AUC {m[\'auc\']:.3f} | sensitivity {m[\'sensitivity\']:.2f} | "\n            f"specificity {m[\'specificity\']:.2f} | accuracy {m[\'accuracy\']:.2f} "\n            f"(threshold {m[\'threshold\']:.2f}, {m[\'n_patients\']} patients, {m[\'n_pdac\']} PDAC)")\n\n\ndef predict_probs(model, dataset, device, batch_size=64, workers=0):\n    """Run the model over a CropDataset. Returns (PDAC probability, patient number) per picture."""\n    import torch  # imported here so the scoring functions above work without torch\n\n    loader = torch.utils.data.DataLoader(\n        dataset, batch_size=batch_size, shuffle=False, num_workers=workers)\n    model.eval()\n    probs, patients = [], []\n    with torch.no_grad():\n        for x, _, patient in loader:\n            scores = torch.softmax(model(x.to(device)), dim=1)[:, 1]  # chance of PDAC\n            probs.append(scores.cpu().numpy())\n            patients.append(patient.numpy())\n    return np.concatenate(probs), np.concatenate(patients)\n', 'cnn_model.py': '"""Small CNN trained from scratch, with 512-dimensional retrieval features."""\nimport os\nimport torch\nfrom torch import nn\n\n\nclass SimpleCNNClassifier(nn.Module):\n    def __init__(self):\n        super().__init__()\n        blocks = []\n        incoming = 3\n        for outgoing in (32, 64, 128, 256):\n            blocks.append(nn.Sequential(\n                nn.Conv2d(incoming, outgoing, 3, padding=1, bias=False),\n                nn.BatchNorm2d(outgoing), nn.ReLU(inplace=False), nn.MaxPool2d(2)))\n            incoming = outgoing\n        self.features = nn.Sequential(*blocks)\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.embed = nn.Sequential(nn.Linear(256, 512), nn.ReLU())\n        self.dropout = nn.Dropout(0.3)\n        self.out = nn.Linear(512, 2)\n\n    def embedding(self, x):\n        return self.embed(self.pool(self.features(x)).flatten(1))\n\n    def forward(self, x):\n        return self.out(self.dropout(self.embedding(x)))\n\n    @property\n    def gradcam_layer(self):\n        return self.features[-1][0]\n\n\ndef save_checkpoint(model, path, **notes):\n    os.makedirs(os.path.dirname(path) or \'.\', exist_ok=True)\n    torch.save({\'arch\': \'simplecnn_512\', \'state_dict\': model.state_dict(), **notes}, path)\n\n\ndef load_classifier(path, device=\'cpu\'):\n    checkpoint = torch.load(path, map_location=device, weights_only=True)\n    if checkpoint.get(\'arch\') != \'simplecnn_512\':\n        raise ValueError(\'Expected a Simple CNN checkpoint\')\n    model = SimpleCNNClassifier()\n    model.load_state_dict(checkpoint[\'state_dict\'])\n    model.to(device).eval()\n    return model, checkpoint\n'}
for name, text in SOURCES.items():
    (WORK / name).write_text(text)
SPLITS_TEXT = 'patient_id,split,label\n100000,train,0\n100001,test,0\n100002,train,1\n100003,val,1\n100004,train,0\n100005,train,1\n100006,train,0\n100007,val,0\n100008,train,0\n100009,train,1\n100010,train,0\n100011,train,1\n100012,train,0\n100013,test,0\n100014,train,0\n100015,train,0\n100016,train,1\n100017,val,0\n100018,train,0\n100019,test,0\n100020,test,0\n100021,val,1\n100022,train,0\n100023,test,0\n100024,val,0\n100025,train,0\n100026,val,0\n100027,val,0\n100028,train,1\n100029,test,0\n100030,test,1\n100031,test,0\n100032,train,0\n100033,train,1\n100034,val,1\n100035,train,0\n100036,train,0\n100037,test,0\n100038,train,0\n100039,train,0\n100040,train,0\n100041,train,0\n100042,train,0\n100043,train,1\n100044,train,0\n100045,train,0\n100046,test,0\n100047,val,0\n100048,train,0\n100049,test,0\n100050,train,1\n100051,train,0\n100052,train,0\n100053,test,0\n100054,train,0\n100055,train,0\n100056,train,1\n100057,train,0\n100058,train,1\n100059,train,0\n100060,train,1\n100061,train,0\n100062,val,0\n100063,train,0\n100064,train,0\n100065,test,1\n100066,train,0\n100067,test,0\n100068,train,0\n100069,train,0\n100070,test,0\n100071,train,0\n100072,train,0\n100073,train,0\n100074,train,1\n100075,train,0\n100076,train,0\n100077,train,0\n100078,train,0\n100079,train,0\n100080,train,0\n100081,train,0\n100082,train,1\n100083,val,0\n100084,train,0\n100085,train,0\n100086,train,1\n100087,train,0\n100088,train,0\n100089,train,0\n100090,train,0\n100091,val,1\n100092,train,0\n100093,test,1\n100094,train,0\n100095,train,0\n100096,train,1\n100097,train,0\n100098,train,0\n100099,train,0\n100100,train,1\n100101,train,1\n100102,train,1\n100103,train,0\n100104,train,0\n100105,train,0\n100106,train,0\n100107,test,1\n100108,train,0\n100109,train,0\n100110,train,1\n100111,train,0\n100112,train,0\n100113,val,1\n100114,train,0\n100115,train,0\n100116,train,1\n100117,val,0\n100118,train,0\n100119,train,0\n100120,train,0\n100121,val,0\n100122,train,0\n100123,test,0\n100124,test,1\n100125,train,0\n100126,train,1\n100127,test,1\n100128,val,0\n100129,train,0\n100130,train,1\n100131,train,0\n100132,val,0\n100133,train,0\n100134,train,1\n100135,train,1\n100136,train,0\n100137,train,0\n100138,test,1\n100139,train,0\n100140,train,1\n100141,test,0\n100142,train,0\n100143,test,1\n100144,train,1\n100145,train,0\n100146,train,0\n100147,train,0\n100148,train,0\n100149,train,0\n100150,test,1\n100151,train,0\n100152,train,0\n100153,train,1\n100154,val,0\n100155,train,0\n100156,train,0\n100157,train,1\n100158,train,0\n100159,val,0\n100160,train,0\n100161,train,1\n100162,train,1\n100163,val,1\n100164,train,1\n100165,train,0\n100166,train,1\n100167,train,1\n100168,train,1\n100169,val,0\n100170,train,0\n100171,test,0\n100172,val,0\n100173,train,0\n100174,train,0\n100175,train,0\n100176,train,1\n100177,train,0\n100178,val,0\n100179,train,0\n100180,train,1\n100181,train,0\n100182,val,0\n100183,train,1\n100184,train,0\n100185,train,0\n100186,train,0\n100187,train,1\n100188,test,0\n100189,train,0\n100190,train,0\n100191,train,0\n100192,train,0\n100193,train,0\n100194,train,1\n100195,train,0\n100196,train,1\n100197,train,1\n100198,train,0\n100199,train,0\n100200,train,1\n100201,test,0\n100202,train,0\n100203,train,0\n100204,train,1\n100205,test,1\n100206,test,0\n100207,train,0\n100208,train,1\n100209,train,0\n100210,test,0\n100211,test,0\n100212,test,0\n100213,train,1\n100214,test,0\n100215,val,0\n100216,val,0\n100217,train,1\n100218,train,0\n100219,val,1\n100220,train,1\n100221,train,0\n100222,val,0\n100223,train,0\n100224,val,0\n100225,val,0\n100226,train,1\n100227,val,0\n100228,train,0\n100229,train,1\n100230,train,0\n100231,train,0\n100232,train,0\n100233,test,0\n100234,train,1\n100235,train,1\n100236,train,0\n100237,train,0\n100238,train,0\n100239,train,0\n100240,train,0\n100241,train,0\n100242,train,0\n100243,train,1\n100244,train,1\n100245,test,1\n100246,test,1\n100247,train,1\n100248,test,1\n100249,train,0\n100250,train,0\n100251,train,0\n100252,train,0\n100253,train,0\n100254,train,0\n100255,train,1\n100256,val,0\n100257,train,1\n100258,train,0\n100259,train,1\n100260,train,0\n100261,test,1\n100262,train,1\n100263,train,0\n100264,train,0\n100265,train,1\n100266,val,1\n100267,train,1\n100268,train,0\n100269,val,0\n100270,val,0\n100271,train,0\n100272,val,0\n100273,val,0\n100274,train,0\n100275,test,1\n100276,train,1\n100277,train,0\n100278,test,1\n100279,train,0\n100280,train,0\n100281,train,0\n100282,train,0\n100283,val,0\n100284,train,0\n100285,train,1\n100286,train,0\n100287,test,0\n100288,test,0\n100289,test,0\n100290,test,1\n100291,train,0\n100292,train,0\n100293,test,0\n100294,test,0\n100295,train,1\n100296,val,1\n100297,test,0\n100298,train,0\n100299,train,0\n100300,train,0\n100301,train,0\n100302,train,0\n100303,train,0\n100304,train,0\n100305,train,0\n100306,val,0\n100307,val,0\n100308,train,0\n100309,test,0\n100310,train,0\n100311,train,0\n100312,train,1\n100313,train,0\n100314,train,0\n100315,train,0\n100316,test,1\n100317,val,0\n100318,train,0\n100319,train,0\n100320,train,0\n100321,test,0\n100322,val,0\n100323,train,1\n100324,train,1\n100325,train,0\n100326,train,0\n100327,train,0\n100328,train,1\n100329,train,0\n100330,train,0\n100331,train,0\n100332,train,0\n100333,train,0\n100334,train,0\n100335,train,1\n100336,train,0\n100337,test,0\n100338,train,0\n100339,train,1\n100340,train,0\n100341,train,0\n100342,test,0\n100343,test,0\n100344,train,1\n100345,val,1\n100346,train,0\n100347,train,1\n100348,train,0\n100349,train,0\n100350,train,0\n100351,train,1\n100352,train,0\n100353,test,0\n100354,val,0\n100355,val,1\n100356,val,1\n100357,train,0\n100358,train,0\n100359,train,0\n100360,test,0\n100361,train,0\n100362,train,0\n100363,train,0\n100364,train,0\n100365,test,0\n100366,test,0\n100367,train,0\n100368,train,1\n100369,train,1\n100370,train,0\n100371,train,0\n100372,test,1\n100373,train,1\n100374,train,0\n100375,test,0\n100376,test,0\n100377,train,0\n100378,train,0\n100379,train,1\n100380,train,0\n100381,train,1\n100382,train,0\n100383,train,0\n100384,train,0\n100385,train,1\n100386,val,1\n100387,test,0\n100388,train,0\n100389,train,0\n100390,val,0\n100391,train,0\n100392,train,0\n100393,train,0\n100394,train,1\n100395,val,0\n100396,test,0\n100397,val,1\n100398,train,0\n100399,train,0\n100400,train,0\n100401,test,0\n100402,train,1\n100403,train,1\n100404,test,1\n100405,test,1\n100406,train,0\n100407,train,1\n100408,train,0\n100409,train,1\n100410,train,0\n100411,train,0\n100412,val,0\n100413,val,1\n100414,train,0\n100415,test,0\n100416,train,1\n100417,train,0\n100418,val,1\n100419,test,1\n100420,train,0\n100421,train,0\n100422,train,0\n100423,train,0\n100424,train,1\n100425,test,0\n100426,train,1\n100427,train,1\n100428,train,0\n100429,train,0\n100430,train,1\n100431,train,1\n100432,val,0\n100433,train,0\n100434,train,0\n100435,train,1\n100436,val,1\n100437,train,0\n100438,train,1\n100439,train,0\n100440,train,1\n100441,train,0\n100442,train,0\n100443,train,0\n100444,train,0\n100445,test,0\n100446,train,0\n100447,test,0\n100448,train,0\n100449,test,0\n100450,train,0\n100451,train,0\n100452,val,0\n100453,train,0\n100454,train,0\n100455,train,0\n100456,train,0\n100457,train,0\n100458,train,0\n100459,train,0\n100460,train,0\n100461,train,0\n100462,test,1\n100463,train,1\n100464,train,1\n100465,val,0\n100466,train,0\n100467,train,1\n100468,train,0\n100469,train,0\n100470,train,0\n100471,test,0\n100472,train,0\n100473,test,0\n100474,train,1\n100475,train,0\n100476,train,0\n100477,train,1\n100478,val,0\n100479,train,1\n100480,train,0\n100481,train,0\n100482,test,0\n100483,train,0\n100484,val,0\n100485,train,0\n100486,train,0\n100487,train,0\n100488,train,0\n100489,train,0\n100490,train,0\n100491,train,0\n100492,train,1\n100493,train,0\n100494,train,0\n100495,train,0\n100496,train,1\n100497,val,0\n100498,train,1\n100499,train,0\n100500,train,0\n100501,train,0\n100502,train,0\n100503,val,1\n100504,train,0\n100505,train,0\n100506,train,1\n100507,train,0\n100508,train,1\n100509,train,0\n100510,train,0\n100511,train,0\n100512,train,1\n100513,train,0\n100514,train,0\n100515,test,1\n100516,val,0\n100517,train,0\n100518,train,0\n100519,train,0\n100520,train,1\n100521,train,0\n100522,train,0\n100523,train,0\n100524,train,0\n100525,train,0\n100526,train,1\n100527,val,0\n100528,train,1\n100529,val,0\n100530,train,1\n100531,train,0\n100532,train,0\n100533,train,1\n100534,train,0\n100535,train,0\n100536,test,0\n100537,val,1\n100538,val,0\n100539,train,0\n100540,test,0\n100541,train,0\n100542,train,1\n100543,train,0\n100544,train,1\n100545,train,1\n100546,train,0\n100547,test,0\n100548,test,0\n100549,train,0\n100550,train,0\n100551,train,0\n100552,val,0\n100553,test,0\n100554,val,1\n100555,train,1\n100556,train,0\n100557,train,1\n100558,val,0\n100559,test,0\n100560,val,0\n100561,test,0\n100562,test,0\n100563,train,1\n100564,train,1\n100565,test,1\n100566,train,0\n100567,train,0\n100568,test,0\n100569,test,0\n100570,train,0\n100571,train,0\n100572,train,0\n100573,train,0\n100574,val,1\n100575,train,0\n100576,train,1\n100577,train,0\n100578,train,0\n100579,train,0\n100580,train,0\n100581,train,0\n100582,val,0\n100583,train,1\n100584,train,0\n100585,train,0\n100586,train,0\n100587,train,0\n100588,test,1\n100589,train,1\n100590,train,0\n100591,train,0\n100592,train,0\n100593,train,0\n100594,train,0\n100595,train,0\n100596,train,0\n100597,train,0\n100598,test,0\n100599,val,1\n100600,train,0\n100601,train,0\n100602,train,1\n100603,train,0\n100604,train,0\n100605,train,0\n100606,test,0\n100607,train,0\n100608,val,1\n100609,train,0\n100610,train,0\n100611,train,1\n100612,val,0\n100613,train,1\n100614,train,0\n100615,test,0\n100616,train,1\n100617,test,1\n100618,train,0\n100619,train,1\n100620,train,0\n100621,train,0\n100622,train,0\n100623,train,1\n100624,train,0\n100625,train,0\n100626,train,0\n100627,train,0\n100628,test,0\n100629,train,1\n100630,train,0\n100631,train,0\n100632,test,0\n100633,train,0\n100634,val,1\n100635,train,0\n100636,train,1\n100637,train,0\n100638,test,0\n100639,train,0\n100640,train,0\n100641,train,0\n100642,train,1\n100643,val,0\n100644,train,0\n100645,train,1\n100646,train,0\n100647,train,1\n100648,val,0\n100649,train,1\n100650,train,0\n100651,train,0\n100652,train,1\n100653,test,0\n100654,val,0\n100655,train,1\n100656,train,0\n100657,val,0\n100658,train,1\n100659,val,0\n100660,test,0\n100661,train,0\n100662,val,0\n100663,train,0\n100664,train,1\n100665,train,1\n100666,train,0\n100667,train,0\n100668,train,1\n100669,train,0\n100670,train,0\n100671,train,0\n100672,train,0\n100673,train,1\n100674,val,0\n100675,train,0\n100676,train,0\n100677,test,0\n100678,test,0\n100679,test,1\n100680,val,0\n100681,train,0\n100682,train,0\n100683,val,0\n100684,val,1\n100685,train,0\n100686,train,0\n100687,train,1\n100688,train,0\n100689,val,0\n100690,test,1\n100691,train,1\n100692,train,0\n100693,test,0\n100694,train,0\n100695,test,1\n100696,train,0\n100697,train,0\n100698,train,1\n100699,val,0\n100700,train,0\n100701,train,0\n100702,train,0\n100703,train,1\n100704,train,0\n100705,test,1\n100706,train,0\n100707,test,0\n100708,test,0\n100709,train,1\n100710,val,0\n100711,train,1\n100712,train,1\n100713,train,1\n100714,test,1\n100715,train,1\n100716,test,0\n100717,val,0\n100718,train,0\n100719,train,0\n100720,val,0\n100721,train,0\n100722,val,1\n100723,test,0\n100724,train,0\n100725,train,1\n100726,train,0\n100727,val,1\n100728,train,1\n100729,val,0\n100730,val,0\n100731,train,0\n100732,val,1\n100733,train,1\n100734,train,1\n100735,train,0\n100736,val,0\n100737,train,0\n100738,train,0\n100739,train,0\n100740,train,1\n100741,train,0\n100742,train,0\n100743,test,1\n100744,train,0\n100745,train,1\n100746,train,0\n100747,val,1\n100748,train,0\n100749,train,0\n100750,train,0\n100751,train,0\n100752,train,0\n100753,val,0\n100754,train,0\n100755,train,1\n100756,train,1\n100757,train,1\n100758,train,0\n100759,val,0\n100760,train,0\n100761,train,0\n100762,train,0\n100763,train,0\n100764,train,0\n100765,val,0\n100766,train,0\n100767,train,0\n100768,train,0\n100769,val,0\n100770,train,0\n100771,train,0\n100772,train,0\n100773,train,0\n100774,train,0\n100775,train,1\n100776,train,1\n100777,test,0\n100778,train,0\n100779,train,0\n100780,test,1\n100781,test,0\n100782,train,0\n100783,train,0\n100784,val,0\n100785,test,0\n100786,test,0\n100787,val,0\n100788,train,0\n100789,test,1\n100790,val,1\n100791,train,0\n100792,train,0\n100793,test,0\n100794,train,0\n100795,test,1\n100796,train,1\n100797,train,0\n100798,train,0\n100799,test,1\n100800,train,0\n100801,test,0\n100802,train,0\n100803,train,1\n100804,train,0\n100805,test,0\n100806,train,0\n100807,train,0\n100808,test,0\n100809,train,1\n100810,train,0\n100811,train,0\n100812,train,0\n100813,train,0\n100814,train,0\n100815,train,1\n100816,test,0\n100817,train,0\n100818,train,0\n100819,train,0\n100820,train,1\n100821,train,0\n100822,train,1\n100823,train,0\n100824,test,1\n100825,train,1\n100826,train,1\n100827,val,0\n100828,train,0\n100829,train,0\n100830,train,0\n100831,val,1\n100832,train,0\n100833,train,0\n100834,train,0\n100835,train,0\n100836,test,0\n100837,train,1\n100838,train,0\n100839,train,0\n100840,train,0\n100841,train,0\n100842,train,1\n100843,train,0\n100844,test,0\n100845,train,0\n100846,val,0\n100847,train,0\n100848,train,0\n100849,train,0\n100850,train,1\n100851,val,0\n100852,train,1\n100853,train,0\n100854,train,0\n100855,train,0\n100856,train,0\n100857,train,0\n100858,train,0\n100859,train,0\n100860,train,1\n100861,train,0\n100862,train,0\n100863,train,0\n100864,test,0\n100865,train,0\n100866,val,0\n100867,train,0\n100868,val,0\n100869,train,0\n100870,train,0\n100871,test,1\n100872,train,0\n100873,train,0\n100874,val,0\n100875,train,1\n100876,train,1\n100877,val,0\n100878,train,0\n100879,val,0\n100880,train,0\n100881,train,0\n100882,train,1\n100883,train,0\n100884,train,1\n100885,test,0\n100886,test,0\n100887,val,0\n100888,val,0\n100889,train,0\n100890,train,0\n100891,val,0\n100892,test,0\n100893,val,0\n100894,test,0\n100895,train,0\n100896,train,0\n100897,train,0\n100898,train,0\n100899,val,0\n100900,train,0\n100901,val,0\n100902,test,0\n100903,train,1\n100904,train,1\n100905,train,1\n100906,train,0\n100907,test,0\n100908,train,1\n100909,val,0\n100910,train,0\n100911,train,0\n100912,test,1\n100913,train,1\n100914,train,0\n100915,train,1\n100916,test,0\n100917,train,0\n100918,train,0\n100919,train,1\n100920,train,0\n100921,train,0\n100922,train,0\n100923,train,0\n100924,train,0\n100925,train,0\n100926,train,1\n100927,train,0\n100928,train,1\n100929,train,0\n100930,train,0\n100931,val,1\n100932,val,1\n100933,train,1\n100934,train,1\n100935,train,0\n100936,train,1\n100937,train,0\n100938,train,0\n100939,train,0\n100940,train,0\n100941,val,1\n100942,train,1\n100943,val,1\n100944,train,0\n100945,val,0\n100946,train,0\n100947,val,0\n100948,train,1\n100949,test,0\n100950,train,1\n100951,val,0\n100952,val,0\n100953,test,0\n100954,train,0\n100955,val,0\n100956,train,1\n100957,val,0\n100958,train,1\n100959,train,1\n100960,train,0\n100961,train,0\n100962,train,0\n100963,train,0\n100964,train,0\n100965,val,1\n100966,val,0\n100967,test,1\n100968,train,0\n100969,test,0\n100970,test,0\n100971,train,0\n100972,train,0\n100973,test,0\n100974,train,1\n100975,test,1\n100976,test,1\n100977,test,0\n100978,train,1\n100979,train,0\n100980,train,0\n100981,train,1\n100982,train,0\n100983,train,1\n100984,train,0\n100985,train,0\n100986,val,0\n100987,train,0\n100988,val,1\n100989,train,0\n100990,train,0\n100991,train,0\n100992,train,0\n100993,train,0\n100994,train,1\n100995,train,1\n100996,train,0\n100997,train,0\n100998,train,0\n100999,test,0\n101000,train,1\n101001,train,0\n101002,train,0\n101003,train,1\n101004,train,0\n101005,train,0\n101006,train,0\n101007,train,1\n101008,val,0\n101009,test,0\n101010,train,0\n101011,train,0\n101012,val,0\n101013,train,1\n101014,test,1\n101015,train,1\n101016,train,0\n101017,train,0\n101018,train,0\n101019,train,1\n101020,train,0\n101021,train,1\n101022,train,0\n101023,train,1\n101024,train,1\n101025,train,0\n101026,train,1\n101027,train,0\n101028,train,0\n101029,train,1\n101030,train,0\n101031,train,1\n101032,train,0\n101033,val,0\n101034,train,0\n101035,test,0\n101036,train,0\n101037,train,0\n101038,train,1\n101039,train,0\n101040,train,0\n101041,val,0\n101042,test,0\n101043,val,1\n101044,train,1\n101045,train,1\n101046,train,1\n101047,val,0\n101048,train,1\n101049,train,1\n101050,test,1\n101051,train,0\n101052,test,0\n101053,train,0\n101054,train,1\n101055,train,0\n101056,train,0\n101057,train,0\n101058,train,0\n101059,train,0\n101060,test,0\n101061,train,0\n101062,val,1\n101063,train,1\n101064,val,1\n101065,train,0\n101066,val,0\n101067,train,0\n101068,test,1\n101069,test,0\n101070,train,0\n101071,train,0\n101072,test,0\n101073,train,0\n101074,test,0\n101075,val,0\n101076,train,0\n101077,train,0\n101078,train,1\n101079,train,0\n101080,train,1\n101081,test,0\n101082,train,0\n101083,val,1\n101084,val,0\n101085,test,1\n101086,test,0\n101087,train,0\n101088,train,0\n101089,val,0\n101090,train,0\n101091,train,1\n101092,train,1\n101093,train,1\n101094,train,0\n101095,train,0\n101096,train,0\n101097,train,1\n101098,test,0\n101099,train,0\n101100,train,0\n101101,train,1\n101102,train,1\n101103,test,1\n101104,train,0\n101105,train,0\n101106,val,1\n101107,test,0\n101108,test,0\n101109,test,0\n101110,train,0\n101111,train,1\n101112,val,1\n101113,train,0\n101114,val,0\n101115,val,0\n101116,train,0\n101117,test,0\n101118,train,1\n101119,train,0\n101120,train,1\n101121,train,0\n101122,train,1\n101123,val,1\n101124,val,0\n101125,val,0\n101126,train,0\n101127,train,1\n101128,test,1\n101129,train,1\n101130,train,0\n101131,val,0\n101132,val,1\n101133,train,1\n101134,val,0\n101135,train,1\n101136,train,0\n101137,train,1\n101138,train,0\n101139,train,1\n101140,train,0\n101141,train,0\n101142,train,1\n101143,train,0\n101144,train,1\n101145,train,0\n101146,train,0\n101147,val,1\n101148,train,0\n101149,val,1\n101150,train,0\n101151,train,1\n101152,val,0\n101153,train,0\n101154,train,0\n101155,train,0\n101156,train,0\n101157,train,0\n101158,test,0\n101159,test,1\n101160,train,1\n101161,train,1\n101162,val,0\n101163,train,1\n101164,train,0\n101165,train,0\n101166,train,0\n101167,train,0\n101168,train,1\n101169,train,1\n101170,test,0\n101171,train,0\n101172,train,0\n101173,test,1\n101174,test,0\n101175,val,1\n101176,val,0\n101177,train,0\n101178,train,1\n101179,test,0\n101180,train,1\n101181,train,0\n101182,val,0\n101183,train,0\n101184,test,0\n101185,train,0\n101186,train,0\n101187,test,0\n101188,val,1\n101189,train,0\n101190,train,0\n101191,train,0\n101192,train,0\n101193,train,0\n101194,train,0\n101195,train,0\n101196,val,1\n101197,train,0\n101198,train,0\n101199,train,0\n101200,test,0\n101201,train,0\n101202,test,0\n101203,train,1\n101204,val,0\n101205,train,1\n101206,train,0\n101207,test,0\n101208,train,0\n101209,train,1\n101210,train,0\n101211,train,0\n101212,val,0\n101213,train,0\n101214,train,0\n101215,test,1\n101216,val,0\n101217,train,1\n101218,train,0\n101219,train,1\n101220,val,1\n101221,train,1\n101222,train,1\n101223,train,0\n101224,val,0\n101225,train,0\n101226,train,0\n101227,val,1\n101228,train,0\n101229,val,1\n101230,val,0\n101231,test,0\n101232,train,1\n101233,train,0\n101234,train,0\n101235,train,0\n101236,train,1\n101237,train,0\n101238,test,1\n101239,test,0\n101240,val,0\n101241,val,0\n101242,train,0\n101243,val,0\n101244,train,1\n101245,train,0\n101246,train,0\n101247,test,1\n101248,val,0\n101249,test,1\n101250,train,0\n101251,train,0\n101252,train,0\n101253,train,1\n101254,train,1\n101255,train,0\n101256,train,0\n101257,train,0\n101258,train,1\n101259,train,0\n101260,test,0\n101261,val,1\n101262,train,0\n101263,train,0\n101264,train,1\n101265,train,0\n101266,train,1\n101267,train,1\n101268,val,0\n101269,test,0\n101270,val,0\n101271,train,1\n101272,train,0\n101273,train,0\n101274,val,1\n101275,train,0\n101276,train,0\n101277,test,0\n101278,train,0\n101279,train,0\n101280,train,0\n101281,train,0\n101282,train,0\n101283,train,0\n101284,train,1\n101285,train,0\n101286,train,1\n101287,train,0\n101288,val,0\n101289,train,0\n101290,val,1\n101291,test,0\n101292,train,0\n101293,train,0\n101294,train,0\n101295,train,1\n101296,train,1\n101297,train,0\n101298,train,0\n101299,train,0\n101300,val,0\n101301,train,0\n101302,train,0\n101303,train,0\n101304,train,0\n101305,train,0\n101306,val,1\n101307,train,0\n101308,test,0\n101309,test,0\n101310,test,0\n101311,train,0\n101312,train,0\n101313,test,1\n101314,test,1\n101315,train,0\n101316,train,1\n101317,train,0\n101318,train,0\n101319,test,0\n101320,train,0\n101321,train,1\n101322,test,1\n101323,train,0\n101324,train,0\n101325,train,0\n101326,train,0\n101327,train,0\n101328,train,0\n101329,train,0\n101330,train,0\n101331,val,1\n101332,train,0\n101333,train,0\n101334,train,0\n101335,train,0\n101336,train,1\n101337,train,0\n101338,train,1\n101339,test,1\n101340,train,1\n101341,train,0\n101342,test,1\n101343,val,0\n101344,train,0\n101345,train,1\n101346,train,0\n101347,test,0\n101348,test,1\n101349,val,0\n101350,test,0\n101351,test,1\n101352,train,1\n101353,val,0\n101354,train,0\n101355,val,1\n101356,val,0\n101357,test,0\n101358,test,0\n101359,train,0\n101360,train,1\n101361,train,0\n101362,train,0\n101363,train,0\n101364,train,0\n101365,train,0\n101366,train,0\n101367,train,1\n101368,train,0\n101369,train,1\n101370,train,0\n101371,val,0\n101372,test,1\n101373,val,0\n101374,train,0\n101375,train,1\n101376,train,1\n101377,train,1\n101378,train,1\n101379,train,0\n101380,val,0\n101381,train,0\n101382,train,0\n101383,train,0\n101384,train,0\n101385,val,0\n101386,val,0\n101387,train,0\n101388,train,0\n101389,train,0\n101390,train,0\n101391,train,1\n101392,train,1\n101393,train,0\n101394,train,1\n101395,train,0\n101396,train,1\n101397,train,0\n101398,train,0\n101399,train,0\n101400,test,0\n101401,test,0\n101402,train,0\n101403,train,1\n101404,train,1\n101405,train,0\n101406,test,1\n101407,val,0\n101408,train,1\n101409,test,1\n101410,train,1\n101411,test,0\n101412,test,1\n101413,train,1\n101414,train,0\n101415,train,1\n101416,train,0\n101417,train,1\n101418,train,1\n101419,train,1\n101420,train,0\n101421,test,1\n101422,train,0\n101423,train,0\n101424,test,0\n101425,train,1\n101426,val,0\n101427,train,1\n101428,train,0\n101429,train,0\n101430,train,0\n101431,train,0\n101432,val,0\n101433,train,0\n101434,val,1\n101435,train,0\n101436,val,1\n101437,val,0\n101438,val,0\n101439,train,0\n101440,val,0\n101441,test,0\n101442,val,0\n101443,val,0\n101444,train,0\n101445,test,0\n101446,train,1\n101447,val,0\n101448,test,0\n101449,train,0\n101450,train,0\n101451,val,1\n101452,train,0\n101453,test,0\n101454,train,1\n101455,train,0\n101456,test,1\n101457,train,0\n101458,test,0\n101459,train,0\n101460,train,0\n101461,train,0\n101462,train,0\n101463,train,0\n101464,train,0\n101465,train,0\n101466,train,0\n101467,train,0\n101468,test,0\n101469,test,0\n101470,train,0\n101471,train,0\n101472,train,1\n101473,train,0\n101474,train,0\n101475,train,1\n101476,val,0\n101477,val,0\n101478,train,1\n101479,train,0\n101480,train,1\n101481,train,0\n101482,train,1\n101483,train,1\n101484,train,0\n101485,val,1\n101486,train,0\n101487,train,0\n101488,train,1\n101489,val,0\n101490,train,0\n101491,train,0\n101492,train,0\n101493,train,0\n101494,val,0\n101495,train,0\n101496,train,1\n101497,train,0\n101498,test,0\n101499,val,0\n101500,train,0\n101501,test,1\n101502,val,0\n101503,train,1\n101504,train,0\n101505,train,1\n101506,test,0\n101507,train,0\n101508,train,1\n101509,train,1\n101510,val,0\n101511,train,0\n101512,train,1\n101513,train,0\n101514,test,0\n101515,test,0\n101516,test,0\n101517,train,1\n101518,val,0\n101519,train,0\n101520,train,0\n101521,train,1\n101522,train,0\n101523,val,0\n101524,test,0\n101525,train,0\n101526,train,0\n101527,train,0\n101528,val,0\n101529,test,0\n101530,val,1\n101531,train,0\n101532,train,1\n101533,train,1\n101534,test,1\n101535,train,0\n101536,train,1\n101537,val,1\n101538,train,1\n101539,train,0\n101540,val,1\n101541,train,1\n101542,test,0\n101543,train,1\n101544,train,0\n101545,train,1\n101546,train,0\n101547,val,1\n101548,test,1\n101549,train,0\n101550,train,1\n101551,train,0\n101552,train,0\n101553,val,0\n101554,train,0\n101555,train,0\n101556,val,0\n101557,train,0\n101558,train,0\n101559,train,0\n101560,train,1\n101561,val,1\n101562,val,0\n101563,train,1\n101564,train,1\n101565,train,1\n101566,train,0\n101567,train,0\n101568,train,0\n101569,val,1\n101570,train,0\n101571,val,0\n101572,train,0\n101573,train,0\n101574,train,1\n101575,train,0\n101576,train,1\n101577,train,0\n101578,train,1\n101579,test,0\n101580,train,0\n101581,train,0\n101582,train,1\n101583,train,0\n101584,train,1\n101585,train,0\n101586,train,0\n101587,val,0\n101588,train,0\n101589,train,1\n101590,train,0\n101591,val,1\n101592,train,1\n101593,train,1\n101594,train,0\n101595,val,0\n101596,train,1\n101597,train,1\n101598,train,1\n101599,train,0\n101600,val,1\n101601,train,0\n101602,test,1\n101603,test,0\n101604,test,1\n101605,test,0\n101606,train,0\n101607,train,0\n101608,val,0\n101609,val,0\n101610,train,0\n101611,val,0\n101612,train,1\n101613,test,0\n101614,train,0\n101615,test,1\n101616,train,1\n101617,train,0\n101618,train,0\n101619,train,0\n101620,val,0\n101621,val,0\n101622,train,0\n101623,val,0\n101624,test,0\n101625,train,1\n101626,train,0\n101627,train,0\n101628,train,0\n101629,val,0\n101630,train,0\n101631,train,0\n101632,test,0\n101633,test,0\n101634,train,1\n101635,train,1\n101636,test,1\n101637,train,1\n101638,val,0\n101639,train,1\n101640,test,1\n101641,train,0\n101642,train,0\n101643,train,0\n101644,test,0\n101645,train,1\n101646,test,0\n101647,test,0\n101648,train,0\n101649,train,0\n101650,train,0\n101651,test,0\n101652,train,0\n101653,train,1\n101654,train,0\n101655,train,0\n101656,test,0\n101657,train,0\n101658,val,1\n101659,train,1\n101660,train,0\n101661,train,1\n101662,train,0\n101663,test,1\n101664,test,0\n101665,train,0\n101666,val,1\n101667,train,0\n101668,train,0\n101669,train,1\n101670,train,0\n101671,test,0\n101672,train,0\n101673,train,0\n101674,train,0\n101675,train,0\n101676,train,0\n101677,train,0\n101678,train,0\n101679,test,1\n101680,train,0\n101681,val,0\n101682,train,0\n101683,train,0\n101684,test,0\n101685,val,0\n101686,train,0\n101687,train,1\n101688,test,0\n101689,train,1\n101690,train,0\n101691,train,0\n101692,train,1\n101693,train,0\n101694,train,1\n101695,train,1\n101696,val,1\n101697,train,1\n101698,train,1\n101699,train,0\n101700,train,0\n101701,train,0\n101702,train,0\n101703,test,0\n101704,test,0\n101705,train,0\n101706,train,0\n101707,train,0\n101708,val,0\n101709,train,0\n101710,train,0\n101711,train,0\n101712,train,0\n101713,train,1\n101714,train,0\n101715,train,0\n101716,test,1\n101717,train,1\n101718,train,0\n101719,test,0\n101720,train,0\n101721,train,1\n101722,train,0\n101723,train,0\n101724,test,0\n101725,train,0\n101726,train,0\n101727,val,0\n101728,train,1\n101729,val,0\n101730,train,0\n101731,train,0\n101732,test,0\n101733,train,1\n101734,train,0\n101735,test,1\n101736,train,0\n101737,train,0\n101738,train,0\n101739,train,0\n101740,test,0\n101741,train,0\n101742,train,0\n101743,train,0\n101744,train,1\n101745,train,0\n101746,train,0\n101747,train,1\n101748,val,0\n101749,train,1\n101750,train,0\n101751,test,1\n101752,test,1\n101753,train,0\n101754,train,0\n101755,train,0\n101756,train,0\n101757,train,1\n101758,train,0\n101759,train,1\n101760,train,0\n101761,test,0\n101762,train,0\n101763,train,0\n101764,val,0\n101765,train,0\n101766,train,0\n101767,test,1\n101768,train,0\n101769,train,0\n101770,val,0\n101771,train,0\n101772,test,0\n101773,val,1\n101774,train,0\n101775,val,0\n101776,train,0\n101777,train,0\n101778,train,0\n101779,val,0\n101780,train,0\n101781,val,0\n101782,test,0\n101783,train,0\n101784,train,0\n101785,train,0\n101786,train,0\n101787,train,1\n101788,train,0\n101789,train,0\n101790,test,0\n101791,train,0\n101792,test,1\n101793,test,0\n101794,train,1\n101795,train,1\n101796,train,1\n101797,val,0\n101798,val,0\n101799,train,0\n101800,train,1\n101801,val,1\n101802,train,0\n101803,train,0\n101804,val,0\n101805,train,0\n101806,train,1\n101807,train,0\n101808,train,1\n101809,val,0\n101810,train,0\n101811,train,0\n101812,train,0\n101813,val,1\n101814,val,0\n101815,val,1\n101816,train,1\n101817,train,0\n101818,train,0\n101819,val,1\n101820,val,0\n101821,train,0\n101822,train,0\n101823,train,0\n101824,train,0\n101825,train,0\n101826,train,0\n101827,train,0\n101828,train,0\n101829,val,1\n101830,val,0\n101831,train,0\n101832,train,1\n101833,test,0\n101834,val,0\n101835,train,1\n101836,train,1\n101837,train,1\n101838,train,1\n101839,train,0\n101840,train,0\n101841,train,1\n101842,train,1\n101843,val,0\n101844,train,0\n101845,train,1\n101846,train,1\n101847,train,1\n101848,test,0\n101849,test,0\n101850,test,0\n101851,val,0\n101852,train,1\n101853,train,0\n101854,val,1\n101855,train,1\n101856,train,1\n101857,train,1\n101858,train,0\n101859,train,0\n101860,test,0\n101861,train,0\n101862,train,0\n101863,train,1\n101864,train,0\n101865,train,0\n101866,train,0\n101867,train,0\n101868,train,0\n101869,test,0\n101870,val,0\n101871,train,0\n101872,train,1\n101873,train,0\n101874,train,0\n101875,train,0\n101876,train,0\n101877,test,1\n101878,train,0\n101879,test,0\n101880,train,1\n101881,train,0\n101882,train,0\n101883,train,0\n101884,train,0\n101885,train,0\n101886,train,1\n101887,val,0\n101888,train,0\n101889,test,0\n101890,train,0\n101891,val,0\n101892,train,0\n101893,test,0\n101894,val,0\n101895,train,1\n101896,train,0\n101897,test,0\n101898,train,1\n101899,train,1\n101900,test,1\n101901,val,0\n101902,train,0\n101903,train,0\n101904,train,0\n101905,train,0\n101906,train,1\n101907,val,0\n101908,train,0\n101909,train,1\n101910,test,0\n101911,train,0\n101912,train,1\n101913,train,0\n101914,train,0\n101915,train,1\n101916,train,0\n101917,train,1\n101918,val,1\n101919,train,0\n101920,train,0\n101921,test,0\n101922,train,0\n101923,val,0\n101924,train,0\n101925,train,0\n101926,train,0\n101927,train,0\n101928,train,0\n101929,test,0\n101930,test,1\n101931,val,1\n101932,test,0\n101933,val,0\n101934,test,0\n101935,train,0\n101936,train,0\n101937,test,0\n101938,val,0\n101939,train,0\n101940,test,1\n101941,test,0\n101942,train,0\n101943,test,0\n101944,test,0\n101945,train,0\n101946,train,0\n101947,val,0\n101948,train,0\n101949,train,0\n101950,train,0\n101951,train,0\n101952,train,1\n101953,train,0\n101954,val,1\n101955,train,1\n101956,train,1\n101957,train,0\n101958,train,0\n101959,train,0\n101960,train,1\n101961,test,0\n101962,train,0\n101963,test,0\n101964,train,1\n101965,val,1\n101966,train,0\n101967,test,0\n101968,train,0\n101969,test,1\n101970,test,0\n101971,train,1\n101972,test,1\n101973,test,0\n101974,train,0\n101975,train,0\n101976,train,1\n101977,train,1\n101978,train,0\n101979,train,0\n101980,train,1\n101981,val,0\n101982,train,0\n101983,test,1\n101984,train,0\n101985,train,0\n101986,test,0\n101987,train,1\n101988,train,1\n101989,train,0\n101990,train,0\n101991,val,0\n101992,train,0\n101993,train,0\n101994,val,1\n101995,train,1\n101996,train,0\n101997,train,0\n101998,train,1\n101999,train,0\n102000,train,0\n102001,val,0\n102002,train,1\n102003,test,1\n102004,val,1\n102005,train,1\n102006,train,1\n102007,val,0\n102008,train,0\n102009,val,1\n102010,val,0\n102011,train,0\n102012,train,0\n102013,train,1\n102014,val,0\n102015,train,0\n102016,train,0\n102017,val,0\n102018,test,0\n102019,train,0\n102020,train,1\n102021,train,0\n102022,test,0\n102023,train,1\n102024,val,1\n102025,train,0\n102026,train,1\n102027,test,0\n102028,train,0\n102029,train,1\n102030,train,0\n102031,train,0\n102032,train,0\n102033,train,1\n102034,val,0\n102035,test,1\n102036,train,0\n102037,train,1\n102038,train,0\n102039,train,1\n102040,train,1\n102041,train,1\n102042,val,0\n102043,train,0\n102044,train,1\n102045,val,1\n102046,train,0\n102047,train,0\n102048,val,1\n102049,val,1\n102050,train,1\n102051,train,1\n102052,val,0\n102053,train,1\n102054,val,1\n102055,train,0\n102056,train,0\n102057,train,1\n102058,train,1\n102059,train,0\n102060,train,0\n102061,train,1\n102062,train,0\n102063,train,1\n102064,train,0\n102065,train,1\n102066,train,1\n102067,train,0\n102068,train,0\n102069,train,0\n102070,train,0\n102071,train,0\n102072,test,1\n102073,train,0\n102074,test,0\n102075,test,0\n102076,val,0\n102077,train,0\n102078,test,0\n102079,train,1\n102080,test,0\n102081,val,0\n102082,train,1\n102083,train,1\n102084,test,1\n102085,val,1\n102086,test,0\n102087,train,0\n102088,test,0\n102089,test,0\n102090,train,1\n102091,train,1\n102092,train,0\n102093,train,0\n102094,train,1\n102095,val,0\n102096,val,0\n102097,train,0\n102098,train,0\n102099,train,0\n102100,train,0\n102101,test,0\n102102,train,1\n102103,train,1\n102104,train,0\n102105,test,1\n102106,train,0\n102107,test,1\n102108,val,0\n102109,val,1\n102110,val,0\n102111,val,1\n102112,train,1\n102113,train,0\n102114,train,0\n102115,test,1\n102116,train,0\n102117,val,0\n102118,train,1\n102119,test,0\n102120,train,0\n102121,train,0\n102122,train,0\n102123,val,0\n102124,train,0\n102125,val,0\n102126,train,0\n102127,test,0\n102128,val,1\n102129,train,0\n102130,train,1\n102131,val,0\n102132,test,1\n102133,train,1\n102134,train,1\n102135,train,0\n102136,train,0\n102137,train,1\n102138,train,1\n102139,train,1\n102140,train,1\n102141,train,1\n102142,train,0\n102143,train,0\n102144,train,0\n102145,train,0\n102146,train,0\n102147,train,1\n102148,train,0\n102149,train,1\n102150,train,0\n102151,train,0\n102152,val,0\n102153,train,0\n102154,train,0\n102155,train,0\n102156,train,1\n102157,test,0\n102158,train,0\n102159,train,1\n102160,train,1\n102161,train,0\n102162,val,0\n102163,val,0\n102164,train,1\n102165,train,0\n102166,val,1\n102167,train,1\n102168,val,0\n102169,test,0\n102170,train,0\n102171,train,1\n102172,test,0\n102173,train,0\n102174,train,0\n102175,test,0\n102176,train,0\n102177,train,0\n102178,train,0\n102179,val,1\n102180,val,0\n102181,train,1\n102182,train,0\n102183,train,0\n102184,test,0\n102185,val,0\n102186,train,0\n102187,test,0\n102188,val,0\n102189,train,0\n102190,train,1\n102191,test,1\n102192,train,0\n102193,test,1\n102194,train,0\n102195,train,1\n102196,train,0\n102197,train,0\n102198,train,0\n102199,train,0\n102200,train,0\n102201,test,0\n102202,train,0\n102203,val,1\n102204,train,1\n102205,val,1\n102206,train,1\n102207,train,0\n102208,train,1\n102209,train,0\n102210,train,0\n102211,train,0\n102212,train,0\n102213,train,0\n102214,train,0\n102215,train,0\n102216,train,1\n102217,train,0\n102218,train,1\n102219,train,0\n102220,val,0\n102221,test,0\n102222,train,0\n102223,val,0\n'
(WORK / 'splits.csv').write_text(SPLITS_TEXT)
sys.path.insert(0, str(WORK))
print('Code ready:', WORK)


## 3. Audit the supplied data before training
Checks every NPZ against the shared split; rejects inconsistent patient IDs, labels, crop types and geometry.
Reports actual coverage. A small/partial dataset is suitable for smoke testing, not final claims.
Arrays and source annotations remain out of Git. No patient is reassigned to a different split.


In [ ]:
import numpy as np, pandas as pd
split_table = pd.read_csv(WORK / 'splits.csv', dtype=str, keep_default_na=False)
assert not split_table.patient_id.duplicated().any(), 'Duplicate patients in shared split'
lookup = split_table.set_index('patient_id')
rows, seen_studies, patient_assignments = [], set(), {}
required = {'crops', 'slice_idx', 'has_lesion', 'label', 'patient_id', 'split', 'study_id', 'window_mm'}
for path in sorted(CROPS.glob('*.npz')):
    if path.name.endswith('.tmp.npz'):
        raise ValueError(f'Incomplete crop file: {path.name}')
    with np.load(path, allow_pickle=False) as item:
        if not required.issubset(item.files):
            raise ValueError(f'Missing NPZ fields: {path.name}')
        pid, study, split = (str(item[k].item()) for k in ['patient_id', 'study_id', 'split'])
        label = int(item['label'].item())
        if pid not in lookup.index or split not in ('train', 'val', 'test'):
            raise ValueError(f'Unknown patient/split: {study}')
        if lookup.loc[pid, 'split'] != split or lookup.loc[pid, 'label'] != str(label):
            raise ValueError(f'Shared split or diagnosis mismatch: {study}')
        if study in seen_studies or path.stem != study or study.split('_')[0] != pid:
            raise ValueError(f'Duplicate or inconsistent study identity: {study}')
        seen_studies.add(study)
        if pid in patient_assignments and patient_assignments[pid] != (split, label):
            raise ValueError(f'Patient crosses splits or labels: {pid}')
        patient_assignments[pid] = (split, label)
        crops, lesion, slices = item['crops'], item['has_lesion'], item['slice_idx']
        if crops.dtype != np.uint8 or crops.ndim != 3 or crops.shape[1:] != (224, 224) or not len(crops):
            raise ValueError(f'Expected nonempty uint8 N x 224 x 224 crops: {study}')
        if lesion.dtype != np.bool_ or lesion.shape != (len(crops),) or slices.shape != (len(crops),):
            raise ValueError(f'Slice metadata mismatch: {study}')
        if len(np.unique(slices)) != len(slices):
            raise ValueError(f'Duplicate slices: {study}')
        window = float(item['window_mm'].item())
        if not np.isfinite(window) or window <= 0:
            raise ValueError(f'Invalid physical crop window: {study}')
        rows.append(dict(study=study, patient=pid, split=split, label=label,
                         slices=len(crops), lesion_slices=int(lesion.sum()), window_mm=window))
audit = pd.DataFrame(rows)
if audit.empty or audit.window_mm.nunique() != 1:
    raise ValueError('Empty data or mixed crop window settings')
print('Studies by split/label:')
print(pd.crosstab(audit.split, audit.label))
print('Patients by split/label:')
print(pd.crosstab(audit.drop_duplicates('patient').split, audit.drop_duplicates('patient').label))
for split in ['train', 'val']:
    if set(audit.loc[audit.split == split, 'label']) != {0, 1}:
        raise ValueError(f'{split} requires both PDAC and Non-PDAC patients')
print('Unlocalised PDAC studies:', len(audit[(audit.label == 1) & (audit.lesion_slices == 0)]))
print('Estimated train+val crop RAM (GiB):', audit[audit.split.isin(['train','val'])].slices.sum()*224*224/1024**3)
audit.to_csv(WORK / 'crop_audit.csv', index=False)


## 4. Inspect example crops
These are cropped greyscale images replicated to 3 channels for ResNet, not colour CT scans.


In [ ]:
import matplotlib.pyplot as plt
examples = [audit[audit.label == label].iloc[0] for label in [0, 1]]
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, row in zip(axes, examples):
    with np.load(CROPS / f'{row.study}.npz', allow_pickle=False) as item:
        ax.imshow(item['crops'][len(item['crops']) // 2], cmap='gray', vmin=0, vmax=255)
    ax.set_title('PDAC' if row.label else 'Non-PDAC'); ax.axis('off')
plt.show()


## 5. Set up the data and measure the untrained baseline
Training and validation only are loaded. The test split is not used for model selection.
The next cell measures the randomly initialized CNN. The full
training cell creates another fresh model, so rerunning the baseline does not contaminate training.


In [ ]:
import random
from dataset import CropDataset
from cnn_model import SimpleCNNClassifier, save_checkpoint
from evaluation import evaluate_patients, patient_scores, predict_probs
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, roc_curve
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
train_ds = CropDataset(CROPS, 'train', mode='train', augment=True)
val_ds = CropDataset(CROPS, 'val', mode='eval')
assert set(train_ds.patient_ids).isdisjoint(set(val_ds.patient_ids))
assert set(train_ds.patient_labels.tolist()) == {0, 1}
print('Training slices/patients:', len(train_ds), len(train_ds.patient_ids))
print('Validation slices/patients:', len(val_ds), len(val_ds.patient_ids))
print('Patient-majority validation accuracy baseline:', max(np.mean(val_ds.patient_labels == 0), np.mean(val_ds.patient_labels == 1)))
print('Random-ranking validation average-precision baseline:', np.mean(val_ds.patient_labels == 1))
DEVICE = 'cuda'
BATCH_SIZE = 16  # reduce to 8 or 4 if CUDA runs out of memory
model = SimpleCNNClassifier().to(DEVICE)
with torch.inference_mode():
    sample = train_ds[0][0].unsqueeze(0).to(DEVICE)
    model.eval()
    assert model.embedding(sample).shape == (1, 512)
    assert model(sample).shape == (1, 2)
print('Model interface verified: embedding 512, logits 2')
p, idx = predict_probs(model, val_ds, DEVICE, batch_size=BATCH_SIZE, workers=0)
print('Untrained-head validation baseline:', evaluate_patients(p, idx, val_ds.patient_labels))


In [ ]:
# Optional smoke test. These weights are discarded before full training.
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    from torch.utils.data import DataLoader
    torch.manual_seed(SEED)
    smoke_model = SimpleCNNClassifier().to(DEVICE)
    smoke_optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=1e-4)
    smoke_loss = torch.nn.CrossEntropyLoss()
    smoke_model.train()
    for step, (x, y, _) in enumerate(DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)):
        if step == 3:
            break
        smoke_optimizer.zero_grad(set_to_none=True)
        loss = smoke_loss(smoke_model(x.to(DEVICE)), y.to(DEVICE))
        loss.backward(); smoke_optimizer.step()
        print(f'Smoke batch {step+1}/3: loss={float(loss.detach()):.4f}', flush=True)
    del smoke_model, smoke_optimizer
    torch.cuda.empty_cache()
    print('Smoke check completed. No accuracy claim; weights discarded.')


## 6. Full training — eight epochs
Defaults: scratch Simple CNN, batch 16, learning rate 1e-4, AdamW, weighted cross-entropy,
seed 0. Saves latest and best checkpoints to Google Drive each epoch. Reducing
batch size to 8 or 4 is fine if CUDA runs out of memory.

An optional three-batch smoke test appears before this cell. Full training starts
from a fresh model and does not reuse smoke-test weights. Do not use Run all if
you only want the smoke test: stop before the full training cell.


In [ ]:
from torch.utils.data import DataLoader
EPOCHS = 8
MAX_BATCHES = None  # full training
LR = 1e-4
from datetime import datetime, timezone
RUN = Path('/content/drive/MyDrive/PanVision/training_runs') / datetime.now(timezone.utc).strftime('simplecnn_%Y%m%dT%H%M%S_%fZ')
RUN.mkdir(parents=True, exist_ok=False)
CHECKPOINT = RUN / 'simplecnn_classifier.pt'
if CHECKPOINT.exists():
    raise RuntimeError('Checkpoint exists. Select a new RUN directory to avoid accidental overwrite.')
# Fresh initialization and seed: smoke-test weights are never reused.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
del model
torch.cuda.empty_cache()
model = SimpleCNNClassifier().to(DEVICE)
weights = torch.tensor([1., train_ds.pos_weight()], device=DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
history, best_auc = [], -1.
config = dict(architecture='simplecnn_512', pretrained=False, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED, max_batches=MAX_BATCHES,
              experiment='annotation-assisted reference crops; not deployment evaluation',
              preprocessing='144mm fixed crop; 224x224 uint8; ImageNet normalization',
              shared_split_sha256=hashlib.sha256(SPLITS_TEXT.encode()).hexdigest())
(RUN / 'config.json').write_text(json.dumps(config, indent=2))
audit.to_csv(RUN / 'crop_audit.csv', index=False)
print('Persistent run folder:', RUN, flush=True)
for epoch in range(1, EPOCHS + 1):
    model.train(); total_loss, seen = 0., 0
    for b, (x, y, _) in enumerate(loader):
        if MAX_BATCHES is not None and b >= MAX_BATCHES:
            break
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE))
        loss.backward(); optimizer.step()
        total_loss += float(loss.detach()) * len(y); seen += len(y)
    if not seen:
        raise RuntimeError('No training batches executed')
    probs, indices = predict_probs(model, val_ds, DEVICE, batch_size=BATCH_SIZE, workers=0)
    metrics = evaluate_patients(probs, indices, val_ds.patient_labels)
    scores, labels = patient_scores(probs, indices, val_ds.patient_labels)
    metrics.update(epoch=epoch, train_loss=total_loss/seen,
                   average_precision=float(average_precision_score(labels, scores)))
    history.append(metrics); print(metrics, flush=True)
    if metrics['auc'] > best_auc:
        best_auc = metrics['auc']
        fpr, tpr, thresholds = roc_curve(labels, scores)
        valid = np.isfinite(thresholds) & (thresholds >= 0) & (thresholds <= 1)
        threshold = float(thresholds[np.flatnonzero(valid)[np.argmax((tpr-fpr)[valid])]])
        save_checkpoint(model, str(CHECKPOINT), epoch=epoch, val_auc=best_auc, threshold=threshold,
                        seed=SEED, smoke_test=MAX_BATCHES is not None, architecture='simplecnn_512',
                        experiment='reference-mask-assisted crops; not deployment evaluation',
                        window_mm=float(audit.window_mm.iloc[0]),
                        aggregation='mean slice probability per patient',
                        splits_sha256=hashlib.sha256(SPLITS_TEXT.encode()).hexdigest())
    torch.save(dict(epoch=epoch, state_dict=model.state_dict(), optimizer=optimizer.state_dict(),
                    best_val_auc=best_auc, config=config), RUN / 'simplecnn_latest.pt')
    (RUN / 'history.json').write_text(json.dumps(history, indent=2))
print('Best checkpoint:', CHECKPOINT)


## 7. Reload the best model and export results
Use the validation-selected threshold only on future held-out evaluation; these validation metrics
are optimistic because the same validation set selected the checkpoint and threshold.
The checkpoint supports `embedding()` and `gradcam_layer`. KNN embeddings must be regenerated
using this checkpoint and exactly the same crop/normalization/pooling recipe.


In [ ]:
from cnn_model import load_classifier
best_model, notes = load_classifier(str(CHECKPOINT), DEVICE)
probs, indices = predict_probs(best_model, val_ds, DEVICE, batch_size=BATCH_SIZE, workers=0)
scores, labels = patient_scores(probs, indices, val_ds.patient_labels)
metrics = evaluate_patients(probs, indices, val_ds.patient_labels, threshold=notes['threshold'])
metrics['average_precision'] = float(average_precision_score(labels, scores))
metrics['f1'] = float(f1_score(labels, scores >= notes['threshold'], zero_division=0))
metrics['confusion_matrix_tn_fp_fn_tp'] = confusion_matrix(labels, scores >= notes['threshold'], labels=[0,1]).ravel().tolist()
print(metrics)
(RUN / 'validation_metrics.json').write_text(json.dumps(metrics, indent=2))
pd.DataFrame({'patient_id': val_ds.patient_ids, 'label': labels, 'pdac_score': scores}).to_csv(RUN / 'validation_predictions.csv', index=False)
import zipfile
bundle = RUN / 'simplecnn_training_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [CHECKPOINT, RUN/'history.json', RUN/'validation_metrics.json', RUN/'validation_predictions.csv', WORK/'crop_audit.csv', WORK/'cnn_model.py', WORK/'splits.csv', RUN/'config.json']:
        archive.write(path, arcname=path.name)
from google.colab import files
files.download(str(bundle))


## Before a final training/evaluation claim
- Inspect crops for clipping, missed pancreas, and lesion-centred fallbacks.
- Use validation/test slice selection that does not consult lesion masks or diagnosis.
- Resolve the mismatch between fixed 144-mm crops here and tight bounding-box crops in the older explainability exporter.
- Separate study-level and patient-level aggregation and record the intended inference unit.
- Evaluate the held-out test set only after finalizing training and threshold selection.
- Report uncertainty and class-specific metrics, not accuracy alone.
- Download checkpoints before the Colab runtime disconnects. Colab local storage is temporary.
